# ArticleTagging — Setup & Benchmark (Qwen3-VL-8B)
Run this notebook on Lightning AI to set up the environment and benchmark Qwen3-VL-8B.

## 1. Clone Repo & Run Setup

In [ ]:
# Run this cell to pull latest changes (preserves data/ and models/)
import os
os.chdir("/kaggle/working/ArticleTagging")
!git stash
!git pull --rebase
!git stash pop 2>/dev/null; true
!pip install -e . -q 2>&1 | tail -1
!git log --oneline -3

In [1]:
import os
os.chdir("/kaggle/working")
!rm -rf ArticleTagging
!git clone https://github.com/Vutiot/ArticleTagging.git
os.chdir("/kaggle/working/ArticleTagging")
!pwd

Cloning into 'ArticleTagging'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 247 (delta 84), reused 245 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 183.21 KiB | 3.33 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/kaggle/working/ArticleTagging


In [2]:
!bash scripts/lightning_setup.sh

=== ArticleTagging Lightning AI Setup ===

--- GPU Info ---
Tesla T4, 15360 MiB, 580.105.08
Tesla T4, 15360 MiB, 580.105.08
PyTorch not yet installed

--- Installing Unsloth ---

--- Installing ArticleTagging ---
    Uninstalling article-tagging-0.1.0:
      Successfully uninstalled article-tagging-0.1.0

--- Installing Claude Code ---
  Claude Code already installed: 2.1.86 (Claude Code)

--- Verifying imports ---
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-28 22:26:38.367595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774736798.391854     605 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774736798.399693     605 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory 

## 2. Verify GPU

In [3]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: Tesla T4
VRAM: 14.6 GB


## 3. Get Dataset
Upload `fashion-data.tar.gz` via the Kaggle sidebar, or download from Kaggle API.

In [4]:
# Option A: If you uploaded fashion-data.tar.gz
# !tar xzf /kaggle/input/fashion-data/fashion-data.tar.gz

# Option B: Download from Kaggle datasets API
!kaggle datasets download paramaggarwal/fashion-product-images-small -p data/raw/fashion/ --unzip

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
License(s): MIT
100%|█████████████████████████████████████████| 565M/565M [00:02<00:00, 219MB/s]



In [5]:
# Verify data
!ls data/processed/fashion/ 2>/dev/null || echo "No processed data yet — run prepare step"
!ls data/raw/fashion/images/ 2>/dev/null | head -5 || echo "No images yet"

No processed data yet — run prepare step
10000.jpg
10001.jpg
10002.jpg
10003.jpg
10004.jpg


## 3.1 Prepare Dataset
Clean, split, and format raw data into training-ready chat format.

In [ ]:
# Step 1: Import Kaggle CSV to JSONL format
from pathlib import Path
from article_tagging.scraping.importers import import_csv, export_jsonl, ImportMapping

mapping = ImportMapping(
    title_field="productDisplayName",
    image_field="id",
    attribute_fields={
        "gender": "gender",
        "masterCategory": "masterCategory",
        "subCategory": "subCategory",
        "articleType": "articleType",
        "baseColour": "baseColour",
        "season": "season",
        "usage": "usage",
    },
)
listings = import_csv(Path("data/raw/fashion/styles.csv"), mapping)
# Fix image paths: CSV has just the id, convert to actual file path
for l in listings:
    if l.image_urls:
        l.image_urls = [f"data/raw/fashion/images/{l.image_urls[0]}.jpg"]
export_jsonl(listings, Path("data/raw/fashion/listings.jsonl"))
print(f"Imported {len(listings)} listings to data/raw/fashion/listings.jsonl")

In [ ]:
# Step 2: Clean, split, and format into chat conversations
!python -c "from article_tagging.cli.main import cli; cli()" prepare \
    --config configs/dataset_fashion.yaml \
    --raw-data data/raw/fashion/listings.jsonl \
    --output-dir data/processed/fashion \
    --image-dir data/raw/fashion/images 2>&1

# Step 3: Create 500-sample test subset for evaluation
import json, random
from pathlib import Path
test = [json.loads(l) for l in Path("data/processed/fashion/test.jsonl").read_text().splitlines() if l.strip()]
random.seed(42)
subset = random.sample(test, 500)
Path("data/processed/fashion/test_500_seed42.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in subset) + "\n"
)
print(f"Created test_500_seed42.jsonl with {len(subset)} samples")

# Verify output
!wc -l data/processed/fashion/*.jsonl

## 4. Train

In [ ]:
# Quick smoke test (10 steps)
!python scripts/benchmark_qwen3vl_8b.py --phase train --max-steps 10

In [ ]:
# Full training (3 epochs)
!python scripts/benchmark_qwen3vl_8b.py --phase train

## 4.1 Monitor Training
Poll the training log to track progress while training runs in the background.

In [ ]:
# Re-run this cell to check training progress
from pathlib import Path

log_path = Path("models/fashion-qwen3vl-8b/training.log")
if log_path.exists():
    lines = log_path.read_text().strip().splitlines()
    for line in lines[-20:]:
        print(line)
    print(f"\n--- {len(lines)} total log lines ---")
else:
    print("Training log not found yet. Start training first.")

# Check for checkpoints
ckpt_dir = Path("models/fashion-qwen3vl-8b")
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob("checkpoint-*"))
    if ckpts:
        print(f"\nCheckpoints: {len(ckpts)}")
        for c in ckpts[-3:]:
            print(f"  {c.name}")

# VRAM usage
import torch
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"\nVRAM: {alloc:.1f} GB allocated, {peak:.1f} GB peak")

## 5. Evaluate

In [ ]:
!python scripts/benchmark_qwen3vl_8b.py --phase eval

In [ ]:
# View results
import json
from pathlib import Path

result_path = Path("reports/v3_qwen3vl_8b/eval_result.json")
if result_path.exists():
    result = json.loads(result_path.read_text())
    print(f"Exact Match: {result['exact_match']:.1%}")
    print("Per-attribute:")
    for attr, acc in result['per_attribute'].items():
        print(f"  {attr}: {acc:.1%}")
else:
    print("No results yet — run eval first")

## 6. Latency Benchmark (vLLM)

In [ ]:
!python scripts/benchmark_qwen3vl_8b.py --phase latency